# Session 12: 32 virtual GPUs and ZeRO-1/2/3

This CPU-friendly notebook creates 32 logical ranks, runs a small PyTorch model for correctness, and makes the ZeRO memory and communication trade-offs explicit. Modeled communication is not a physical GPU benchmark.

## Conceptual setup

Ordinary data parallelism replicates the full model, gradients, FP32 master weights, and Adam moments on every rank. ZeRO progressively removes this redundancy: ZeRO-1 shards optimizer states; ZeRO-2 also shards gradients; ZeRO-3 also shards parameters. Adam-style training is approximately 16 bytes per parameter before sharding: 2 + 2 + 4 + 4 + 4 bytes.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from zero_simulator import (
    STAGES, VirtualCluster, build_experiment_rows, correctness_check, human_bytes,
    mlp_parameter_count, shard_sizes, stage_shard_summary, benchmark_cpu_step,
)

WORLD_SIZE = 32
INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM = 64, 1024, 64
BATCH_SIZE = 128
PARAMETERS = mlp_parameter_count(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM)
ACTIVATIONS = 2 * 1024**2
cluster = VirtualCluster(PARAMETERS, WORLD_SIZE)
print(f'{WORLD_SIZE} logical GPU ranks; {PARAMETERS:,} trainable parameters')
print('logical ranks created:', len(cluster.ranks))

## 1. Memory and communication table

The arithmetic workload is held constant. The state ownership and collective traffic change by stage.

In [ ]:
rows = build_experiment_rows(
    parameters=PARAMETERS, world_size=WORLD_SIZE, bandwidth_gbps=5.0,
    activation_bytes_per_rank=ACTIVATIONS, batch_size=BATCH_SIZE,
    input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM,
    compute_tflops=1.0,
)
results = pd.DataFrame(rows)
view = results[['stage', 'model_state_bytes_per_rank', 'peak_bytes_per_rank',
    'total_cluster_state_bytes', 'communication_total_bytes_per_rank',
    'estimated_forward_backward_flops_cluster', 'estimated_compute_ms_at_configured_tflops',
    'estimated_step_ms']].copy()
view['model_state_mib'] = view['model_state_bytes_per_rank'] / 2**20
view['peak_mib'] = view['peak_bytes_per_rank'] / 2**20
view['communication_mib'] = view['communication_total_bytes_per_rank'] / 2**20
display(view.round(2))

## 2. Memory ladder

The idealized per-rank model-state formulas are: ZeRO-0 = 16P; ZeRO-1 = 4P + 12P/N; ZeRO-2 = 2P + 14P/N; ZeRO-3 = 16P/N plus temporary layer buffers.

In [ ]:
labels = ['ZeRO-0', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3']
colors = ['#6b7280', '#2563eb', '#059669', '#d97706']
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(labels, results['model_state_bytes_per_rank'] / 2**20, color=colors)
axes[0].set(title='Model state per logical rank', ylabel='MiB')
axes[0].grid(axis='y', alpha=0.25)
axes[1].bar(labels, results['total_cluster_state_bytes'] / 2**30, color=colors)
axes[1].set(title='Total cluster model state', ylabel='GiB')
axes[1].grid(axis='y', alpha=0.25)
plt.show()
for stage, value in zip(labels, results['model_state_bytes_per_rank']):
    print(f'{stage}: {human_bytes(value)} per rank')

## 3. Shard ownership and uneven layers

A shard is an ownership assignment, not necessarily a perfect layer split. Remainders and differently sized layers can make one rank slightly larger.

In [ ]:
sizes = shard_sizes(17, WORLD_SIZE)
print('17 elements across 32 ranks:', sizes[:8], '...', sizes[-3:], 'sum =', sum(sizes))
layer_sizes = [40_000 + (i % 5) * 7_000 for i in range(35)]
summary = stage_shard_summary(PARAMETERS, WORLD_SIZE, layer_sizes)
print('parameter shard range:', summary['parameter_shard_min'], 'to', summary['parameter_shard_max'])
print('layer-group shard range:', summary['layer_shard_min'], 'to', summary['layer_shard_max'])

## 4. Communication versus computation

ZeRO-0/1/2 synchronize gradients. ZeRO-3 adds parameter all-gather before layer computation. The arithmetic work is flat in this model because ZeRO changes state ownership, not the mathematical training objective.

In [ ]:
comm = results[['stage', 'communication_gradient_collective_bytes',
    'communication_parameter_all_gather_bytes', 'communication_total_bytes_per_rank',
    'communication_estimated_ms', 'estimated_step_ms']].copy()
comm['gradient_mib'] = comm['communication_gradient_collective_bytes'] / 2**20
comm['parameter_gather_mib'] = comm['communication_parameter_all_gather_bytes'] / 2**20
comm['total_mib'] = comm['communication_total_bytes_per_rank'] / 2**20
display(comm.round(3))
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
axes[0].bar(labels, results['communication_total_bytes_per_rank'] / 2**20, color=colors)
axes[0].set(title='Modeled communication per rank', ylabel='MiB per step')
axes[0].grid(axis='y', alpha=0.25)
axes[1].bar(labels, results['estimated_forward_backward_flops_cluster'] / 1e9, color=colors)
axes[1].set(title='Arithmetic work across cluster', ylabel='GFLOPs per step')
axes[1].grid(axis='y', alpha=0.25)
axes[2].bar(labels, results['estimated_step_ms'], color=colors)
axes[2].set(title='Estimated step cost', ylabel='ms; 1 TFLOP/s assumption')
axes[2].grid(axis='y', alpha=0.25)
plt.show()

The communication estimate uses a ring factor `2(N-1)/N` and a configurable bandwidth. This example uses 5 GB/s deliberately so the effect is visible for a tiny toy model; 450 GB/s can be used for an NVLink-like sensitivity check. The estimated step column assumes a configurable 1 TFLOP/s arithmetic rate only to make the relative communication overhead visible; it is not a claim about this machine's hardware.

## 5. Correctness and CPU context

All stages use the same deterministic Adam update in the toy correctness check. The simulator changes state ownership and communication accounting, not the optimization rule.

In [ ]:
check = correctness_check()
display(pd.DataFrame([check['differences_by_stage']]))
assert check['all_close'], check
print(check['note'])
cpu_timing = benchmark_cpu_step(repeats=3)
print(f"Mean CPU reference step: {cpu_timing['mean_seconds'] * 1000:.2f} ms")

## Conclusion

I understand ZeRO as progressively removing replicated training state. ZeRO-1 is a low-disruption memory improvement, ZeRO-2 saves additional gradient memory, and ZeRO-3 makes the largest models fit by sharding parameters too. ZeRO-3 is not automatically fastest because parameter all-gathers and temporary buffers increase communication sensitivity. The best stage depends on model size, batch size, compute time, bandwidth, and memory per rank.